# IMDB Sentiment Analysis — Visual Analytics & Modeling
This notebook separates the IMDB sentiment analysis pipeline into modular sections with rich data visualizations, charts, and model diagnostics.

## 1. Imports and Configuration

In [ ]:
import re
import html
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
SEED = 42

DATA_PATH = "IMDB Dataset.csv"
OUTPUT_DIR = "outputs"
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

## 2. Helper Functions

In [ ]:
def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z']", " ", text)
    return re.sub(r"\s+", " ", text).lower().strip()


def create_wordcloud(series, title, path, colormap="viridis"):
    words = " ".join(series.astype(str).tolist())
    cloud = WordCloud(
        width=1400,
        height=800,
        background_color="white",
        max_words=180,
        collocations=False,
        colormap=colormap,
    ).generate(words)
    plt.figure(figsize=(12, 7))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=17, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

## 3. Data Loading and Preprocessing

In [ ]:
df = pd.read_csv(DATA_PATH, usecols=["review", "sentiment"])
raw_count = len(df)
df = df.dropna().drop_duplicates().copy()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df.sentiment.isin(["positive", "negative"])].copy()
df["clean_review"] = df.review.map(clean_text)
df = df[df.clean_review.str.len() > 0].copy()

df["word_count"] = df["clean_review"].str.split().str.len()
df["char_count"] = df["clean_review"].str.len()

print(f"Raw rows: {raw_count:,} | Cleaned unique rows: {len(df):,}")
print("Class counts:\n", df.sentiment.value_counts())

## 4. Exploratory Data Analysis (EDA) Charts

In [ ]:
# 4A. Sentiment Class Distribution (Bar Chart + Donut Chart)
counts = df["sentiment"].value_counts()
colors = ["#2ecc71", "#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart with exact count annotations
bars = axes[0].bar(counts.index, counts.values, color=colors, width=0.5, edgecolor="black")
axes[0].set_title("Sentiment Class Counts", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Number of Reviews")
for bar in bars:
    h = bar.get_height()
    axes[0].annotate(
        f"{h:,}",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        fontweight="bold",
    )

# Donut chart for class balance proportion
axes[1].pie(
    counts.values,
    labels=counts.index.str.capitalize(),
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"width": 0.45, "edgecolor": "white"},
    textprops={"fontsize": 12, "fontweight": "bold"},
)
axes[1].set_title("Sentiment Proportion", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(out / "sentiment_distribution.png", dpi=160)
plt.show()
plt.close()

In [ ]:
# 4B. Review Length Distribution by Sentiment (Histogram/KDE + Boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df,
    x="word_count",
    hue="sentiment",
    bins=50,
    kde=True,
    palette={"positive": "#2ecc71", "negative": "#e74c3c"},
    alpha=0.45,
    ax=axes[0],
)
axes[0].set_title("Word Count Distribution by Sentiment", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Words per Review")
axes[0].set_ylabel("Review Count")
axes[0].set_xlim(0, df["word_count"].quantile(0.99))

sns.boxplot(
    data=df,
    x="sentiment",
    y="word_count",
    palette={"positive": "#2ecc71", "negative": "#e74c3c"},
    showfliers=False,
    ax=axes[1],
)
axes[1].set_title("Review Word Count Spread (Outliers Hidden)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Sentiment")
axes[1].set_ylabel("Words per Review")

plt.tight_layout()
plt.savefig(out / "review_length_distribution.png", dpi=160)
plt.show()
plt.close()

## 5. Word Clouds & Top N-Gram Frequency Charts

In [ ]:
# 5A. Positive and Negative Word Clouds
create_wordcloud(
    df.loc[df.sentiment == "positive", "clean_review"],
    "Positive Reviews Word Cloud",
    out / "positive_wordcloud.png",
    colormap="Greens",
)
create_wordcloud(
    df.loc[df.sentiment == "negative", "clean_review"],
    "Negative Reviews Word Cloud",
    out / "negative_wordcloud.png",
    colormap="Reds",
)

In [ ]:
# 5B. Top 15 Most Frequent Words per Sentiment (Horizontal Bar Charts)
from sklearn.feature_extraction.text import CountVectorizer

def plot_top_terms(texts, ax, title, color):
    cv = CountVectorizer(stop_words="english", max_features=15)
    mat = cv.fit_transform(texts)
    freqs = np.asarray(mat.sum(axis=0)).ravel()
    terms = cv.get_feature_names_out()
    order = np.argsort(freqs)
    ax.barh(terms[order], freqs[order], color=color, edgecolor="black")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Frequency")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_top_terms(
    df.loc[df.sentiment == "positive", "clean_review"],
    axes[0],
    "Top 15 Words in Positive Reviews",
    "#2ecc71",
)
plot_top_terms(
    df.loc[df.sentiment == "negative", "clean_review"],
    axes[1],
    "Top 15 Words in Negative Reviews",
    "#e74c3c",
)
plt.tight_layout()
plt.savefig(out / "top_terms_by_sentiment.png", dpi=160)
plt.show()
plt.close()

## 6. Train-Test Split and TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.clean_review,
    df.sentiment,
    test_size=0.2,
    random_state=SEED,
    stratify=df.sentiment,
)

vectorizer = TfidfVectorizer(
    max_features=40000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    sublinear_tf=True,
    stop_words="english",
    dtype=np.float32,
)
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)
print("Training matrix shape:", Xtr.shape)
print("Testing matrix shape:", Xte.shape)

## 7. Model Training, Metric Charts & ROC/PR Curves

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=300, C=2, solver="liblinear", random_state=SEED
    ),
    "SGD Classifier": SGDClassifier(
        loss="hinge", alpha=1e-5, max_iter=20, tol=1e-3, random_state=SEED
    ),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.5),
}

fitted = {}
records = []

for name, clf in classifiers.items():
    print("Training:", name, flush=True)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    records.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, pos_label="positive"),
            "Recall": recall_score(y_test, pred, pos_label="positive"),
            "F1-score": f1_score(y_test, pred, pos_label="positive"),
        }
    )
    fitted[name] = clf

results = pd.DataFrame(records).sort_values(["F1-score", "Accuracy"], ascending=False)
results.to_csv(out / "model_comparison.csv", index=False)
display(results)

# Grouped Bar Chart with Value Labels
plot_data = results.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(11, 6))
ax = sns.barplot(data=plot_data, x="Model", y="Score", hue="Metric", palette="deep")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8.5, padding=2)
plt.ylim(0.7, 1.03)
plt.title("Model Performance Comparison Across Metrics", fontsize=14, fontweight="bold")
plt.xticks(rotation=10, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(out / "model_comparison.png", dpi=160)
plt.show()
plt.close()

In [ ]:
# 7B. ROC Curve & Precision-Recall Curve Across All Models
y_test_bin = (y_test == "positive").astype(int)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for name, clf in fitted.items():
    if hasattr(clf, "predict_proba"):
        scores = clf.predict_proba(Xte)[:, 1]
    else:
        scores = clf.decision_function(Xte)

    fpr, tpr, _ = roc_curve(y_test_bin, scores)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.4f})")

    prec, rec, _ = precision_recall_curve(y_test_bin, scores)
    pr_auc = auc(rec, prec)
    axes[1].plot(rec, prec, lw=2, label=f"{name} (AUC = {pr_auc:.4f})")

axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_title("ROC Curves", fontsize=14, fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

axes[1].set_title("Precision-Recall Curves", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.savefig(out / "roc_pr_curves.png", dpi=160)
plt.show()
plt.close()

## 8. Best Model Evaluation, Confusion Matrix & Feature Importance Chart

In [ ]:
best_name = results.iloc[0]["Model"]
best = fitted[best_name]
pred = best.predict(Xte)
labels = ["negative", "positive"]

# Confusion Matrix Heatmap (Counts + Percentages)
cm = confusion_matrix(y_test, pred, labels=labels)
cm_pct = cm / cm.sum() * 100
annot = np.array(
    [[f"{c:,}\n({p:.1f}%)" for c, p in zip(row_c, row_p)] for row_c, row_p in zip(cm, cm_pct)]
)

plt.figure(figsize=(6.5, 5.5))
sns.heatmap(
    cm, annot=annot, fmt="", cmap="Blues", xticklabels=labels, yticklabels=labels, cbar=False
)
plt.title(f"Confusion Matrix — {best_name}", fontsize=14, fontweight="bold")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.tight_layout()
plt.savefig(out / "confusion_matrix.png", dpi=160)
plt.show()
plt.close()

report = (
    "Best model: "
    + best_name
    + "\n\n"
    + classification_report(y_test, pred, digits=4)
)
report += "\n\nModel comparison:\n" + results.to_string(index=False)
(out / "classification_report.txt").write_text(report, encoding="utf-8")
print(report)

In [ ]:
# 8B. Top Predictive Positive vs. Negative Features Chart (Linear Models)
if hasattr(best, "coef_"):
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = best.coef_.ravel()
    top_n = 15

    top_pos_idx = np.argsort(coefs)[-top_n:]
    top_neg_idx = np.argsort(coefs)[:top_n]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].barh(feature_names[top_pos_idx], coefs[top_pos_idx], color="#2ecc71", edgecolor="black")
    axes[0].set_title(f"Top {top_n} Positive Sentiment Features ({best_name})", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Model Weight")

    axes[1].barh(feature_names[top_neg_idx], coefs[top_neg_idx], color="#e74c3c", edgecolor="black")
    axes[1].set_title(f"Top {top_n} Negative Sentiment Features ({best_name})", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Model Weight")

    plt.tight_layout()
    plt.savefig(out / "top_predictive_features.png", dpi=160)
    plt.show()
    plt.close()

## 9. Save Best Model Pipeline

In [ ]:
# Store vectorizer and classifier together so raw reviews can be passed to predict().
pipeline = Pipeline([("tfidf", vectorizer), ("classifier", best)])
joblib.dump(
    {"model": pipeline, "best_model_name": best_name, "classes": list(best.classes_)},
    out / "best_model.pkl",
    compress=3,
)

print("\nBEST MODEL:", best_name)
print(results.to_string(index=False))
print("Saved all outputs to", out.resolve())